# 03 — KPI analysis and business recommendations

This notebook calculates the KPI scorecard, uses SQL for grouped analysis, and performs a statistical comparison of average transaction values between two branches.

In [ ]:
import pandas as pd
import sqlite3
from scipy import stats
from pathlib import Path
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sales = pd.read_csv(ROOT / 'data/processed/sales_clean.csv', parse_dates=['date'])
products = pd.read_csv(ROOT / 'data/raw/products.csv')
df = sales.merge(products, on='product_id')
conn = sqlite3.connect(ROOT / 'data/duka_retail.db')
# Recreate the table in case notebook 01 was not run in this session
sales.to_sql('sales_clean', conn, if_exists='replace', index=False)
products.to_sql('products', conn, if_exists='replace', index=False)

## KPI scorecard

Revenue = sum of net sales; average transaction value = revenue divided by transactions; customer count = distinct customer IDs; discount rate = total discounts divided by gross sales.

In [ ]:
kpis = pd.DataFrame({'KPI':['Net revenue (KES)','Transactions','Average transaction value (KES)','Unique customers','Discount rate'], 'Value':[sales.net_sales_kes.sum(),len(sales),sales.net_sales_kes.mean(),sales.customer_id.nunique(),sales.discount_kes.sum()/sales.gross_sales_kes.sum()]})
kpis

## SQL analysis using pandas read_sql

In [ ]:
branch_sql = '''SELECT branch, COUNT(*) AS transactions, SUM(net_sales_kes) AS revenue_kes FROM sales_clean GROUP BY branch ORDER BY revenue_kes DESC'''
pd.read_sql(branch_sql, conn)

## Statistical test — Nairobi vs Mombasa transaction values

Welch's independent-samples t-test compares mean transaction value without assuming equal variances. A p-value below 0.05 would indicate evidence of a difference at the 5% significance level.

In [ ]:
a = sales.loc[sales.branch=='Nairobi','net_sales_kes']
b = sales.loc[sales.branch=='Mombasa','net_sales_kes']
t_stat, p_value = stats.ttest_ind(a, b, equal_var=False)
print(f'Nairobi mean: KES {a.mean():,.2f}')
print(f'Mombasa mean: KES {b.mean():,.2f}')
print(f't-statistic: {t_stat:.4f}; p-value: {p_value:.4f}')
print('Interpretation:', 'Evidence of a difference' if p_value < 0.05 else 'No statistically significant difference detected at 5%.')
conn.close()